In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
module_path = os.path.abspath(os.path.join('../../../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [3]:
print(sys.path)

['/usr/lib/python311.zip', '/usr/lib/python3.11', '/usr/lib/python3.11/lib-dynload', '', '/data/isa/mlex/venv/lib/python3.11/site-packages', '/data/isa/mlex']


In [4]:
import pandas as pd

In [ ]:
base_path = r'/data/eeg_eyestate/EEG_Eye_State.csv'
label_column = 'eyeDetection'
df = pd.read_csv(base_path, delimiter=';')


In [ ]:
df

In [ ]:
counts = df[label_column].value_counts()
        
# 4. Calcular a porcentagem
total = counts.sum()
percentages = (counts / total) * 100

# 5. Imprimir os resultados de forma clara
print("\n--- Análise de Classes (eyedetection) ---")
print(f"Total de amostras: {total:,}".replace(",", "."))
print("-" * 35)

for label, count in counts.items():
    percent = percentages.loc[label]
    print(f"Classe {label}: {count:,} amostras ({percent:.2f}%)".replace(",", "."))




In [ ]:
df_analise = pd.DataFrame({
    'Count': counts,
    'Percentage': percentages
})

df_analise.index = df_analise.index.map({
    0: 'Eyes Open (0)',
    1: 'Eyes Closed (1)'
})

# --------------------------------------------------------------------------
# --- FUNÇÃO CORRIGIDA PARA REMOVER TODO O LIXO DO PANDAS TO_LATEX ---
# --------------------------------------------------------------------------

def gerar_latex_analise_classes(df):
    
    # 3.1. Criar um DataFrame formatado com as strings finais
    df_latex = pd.DataFrame({
        'Samples': df['Count'].apply(lambda x: f'{x:,}'.replace(',', '.')),
        'Percentage': df['Percentage'].apply(lambda x: f'{x:.2f}\\%')
    })
    
    caption = "Distribution of Samples by Class in the EEG Eye State Dataset."
    label = "tab:class_distribution"
    column_format = "lrr" 
    
    # Geramos o LaTeX, mas pedimos para NÃO INSERIR CABEÇALHO (header=False)
    # nem linhas (usando default, que é sem linhas para versões antigas)
    latex_string = df_latex.to_latex(
        index_names=False, # Não exibe o nome do índice (Class, no caso)
        header=False,      # Não exibe a linha de cabeçalho
        column_format=column_format, 
        escape=False       # Mantém o '\\%' como %
    )

    # Pegamos apenas as linhas de DADOS geradas, ignorando \begin{tabular} e \end{tabular}
    linhas_dados = latex_string.splitlines()
    corpo_tabela = "\n".join(linhas_dados[1:-1]).strip()

    # Inserimos o conteúdo gerado em um template limpo com comandos booktabs
    latex_final = f"""
\\begin{{table}}[!htbp]
\\small
\\caption{{{caption}}}
\\centering
\\begin{{center}}
\\begin{{tabular}}{{{column_format}}}
\\toprule
Class & Samples (N) & Percentage (\\%) \\\\
\\midrule
{corpo_tabela}
\\bottomrule
\\end{{tabular}}
\\end{{center}}
\\label{{{label}}}
\\end{{table}}
"""
    return latex_final


# --- GERAÇÃO FINAL DO CÓDIGO LaTeX ---
codigo_latex_final = gerar_latex_analise_classes(df_analise)

print("\n\n#################################################")
print("### CÓDIGO LaTeX DA TABELA DE CLASSES (FINAL) ###")
print("#################################################")
print(codigo_latex_final)

In [ ]:
base_path = r'/data/eeg_eyestate/EEG_Eye_State'
cluster_names = ['kmeans', 'gmm', 'agglomerative']
target_column = 'eyeDetection'
group_column = 'GROUP'

total_global_rows = 0 

all_results_df = pd.DataFrame()

for cluster_name in cluster_names:
    path = f'{base_path}_{cluster_name}.csv'


    df = pd.read_csv(path, sep=';', decimal=',')
    total_global_rows = len(df)

    df[target_column] = df[target_column].astype(int)
    distribution_table = pd.crosstab(
            df[group_column], 
            df[target_column], 
            margins=True,
            normalize=False
        )
    distribution_table = distribution_table.rename(columns={
            0: 'Target 0', 
            1: 'Target 1', 
        })
    
    distribution_table = distribution_table.reset_index()
    distribution_table.insert(0, 'Algorithm', cluster_name.lower())
    all_results_df = pd.concat([all_results_df, distribution_table], ignore_index=True)



final_table = all_results_df[all_results_df[group_column] != 'Total Geral'].copy()
final_table = final_table[final_table[group_column] != 'All'].copy()

final_table['% 0'] = (final_table['Target 0'] / total_global_rows) * 100
final_table['% 1'] = (final_table['Target 1'] / total_global_rows) * 100

final_table = final_table[['Algorithm', group_column, 'Target 0', '% 0', 'Target 1', '% 1']]
final_table.columns = ['Algorithm', 'Cluster', '0', '% 0', '1', '% 1']

print(final_table.to_string(index=False))
print("\nAnálise de distribuição consolidada concluída.")




In [ ]:
from io import StringIO

def generate_latex_table(df):
    """Gera a string LaTeX no formato hierárquico solicitado, com contagem e porcentagem."""
    
    latex_string = StringIO()
    latex_string.write(r'\begin{table}[h!]' + '\n')
    latex_string.write(r'\centering' + '\n')
    latex_string.write(r'\caption{Distribution of the Target Column by Cluster. }' + '\n')
    latex_string.write(r'\label{tab:cluster_target_distribution}' + '\n')
    
    # Define o número de colunas: Algorithm + Cluster + (Contagem + %) x 2 = 6 Colunas (c c r r r r)
    latex_string.write(r'\begin{tabular}{ccrr rr}' + '\n') 
    latex_string.write(r'\toprule' + '\n')

    # Cabeçalho Principal (Colspan 4 para as colunas de Targets)
    latex_string.write(r'\textbf{Algorithm} & \textbf{Cluster} & \multicolumn{4}{c}{\textbf{' + target_column + r'}}' + r' \\' + '\n')
    
    # Cabeçalho Secundário (Subdivisão de Contagem)
    latex_string.write(r'\cmidrule(lr){3-6}' + '\n')
    # Novo Cabeçalho: 0 (Contagem) | 0 (%) | 1 (Contagem) | 1 (%)
    latex_string.write(r' & & \multicolumn{2}{c}{\textbf{0}} & \multicolumn{2}{c}{\textbf{1}}' + r' \\' + '\n') 
    latex_string.write(r'\cmidrule(lr){3-4}\cmidrule(lr){5-6}' + '\n')
    latex_string.write(r' & & \multicolumn{1}{c}{Count} & \multicolumn{1}{c}{(\%)} & \multicolumn{1}{c}{Count} & \multicolumn{1}{c}{(\%)}' + r' \\' + '\n')
    latex_string.write(r'\midrule' + '\n')
    
    current_algo = None
    
    # Itera sobre o DataFrame
    for index, row in df.iterrows():
        algo = row['Algorithm']
        cluster = str(row['Cluster'])
        
        # Formata Contagens (com ponto como separador de milhar)
        count_0 = f"{row['0']:,}".replace(',', 'X').replace('.', ',').replace('X', '.')
        count_1 = f"{row['1']:,}".replace(',', 'X').replace('.', ',').replace('X', '.')
        
        # Formata Porcentagens (1 casa decimal)
        percent_0 = f"{row['% 0']:.1f}"
        percent_1 = f"{row['% 1']:.1f}"

        # Verifica se o algoritmo mudou
        if algo != current_algo:
            if current_algo is not None:
                latex_string.write(r'\midrule' + '\n')
                
            num_clusters = df[df['Algorithm'] == algo].shape[0]
            
            # Cria a linha do Algoritmo com \multirow
            latex_string.write(r'\multirow{' + str(num_clusters) + r'}{*}{\textbf{' + algo + r'}}')
            current_algo = algo
        else:
            # Coluna vazia para \multirow
            latex_string.write(r'')
            
        # Linhas de Dados do Cluster (Count | % | Count | %)
        latex_string.write(f" & {cluster} & {count_0} & ({percent_0}) & {count_1} & ({percent_1})" + r' \\' + '\n')
        
    latex_string.write(r'\bottomrule' + '\n')
    latex_string.write(r'\end{tabular}' + '\n')
    latex_string.write(r'\end{table}' + '\n')
    
    return latex_string.getvalue()

latex_output = generate_latex_table(final_table)
print(latex_output)

In [ ]:
import pandas as pd
import os
import sys

# --- CONFIGURAÇÃO ---
base_path = r'/data/eeg_eyestate/EEG_Eye_State'
cluster_names = ['kmeans', 'gmm', 'agglomerative']
target_column = 'eyeDetection'
group_column = 'GROUP'
sets = ['train', 'test']

all_results_df = pd.DataFrame()

print("Iniciando processamento dos arquivos...")

# --- LOOP DE PROCESSAMENTO ---
for cluster_name in cluster_names:
    for data_set in sets:
        path = os.path.join(os.path.dirname(base_path), f'EEG_Eye_State_{cluster_name}_{data_set}.csv')
        
        try:
            df = pd.read_csv(path, sep=';', decimal=',')
        except FileNotFoundError:
            print(f"⚠️ Aviso: Arquivo não encontrado em {path}. Pulando.")
            continue
            
        total_set_rows = len(df)
        df[target_column] = df[target_column].astype(int)
        
        distribution_table = pd.crosstab(df[group_column], df[target_column], margins=True, normalize=False)
        distribution_table_perc = (distribution_table / total_set_rows) * 100
        
        counts_df = distribution_table.rename(columns={0: 'Count_0', 1: 'Count_1', 'All': 'Total_Count'}).reset_index()
        perc_df = distribution_table_perc.rename(columns={0: 'Perc_0', 1: 'Perc_1', 'All': 'Total_Perc'}).reset_index()
        
        temp_df = counts_df.merge(perc_df, on=group_column)
        temp_df.insert(0, 'Set', data_set.capitalize())
        temp_df.insert(0, 'Algorithm', cluster_name.lower())
        
        temp_df = temp_df.loc[temp_df[group_column] != 'All'].copy()
        
        all_results_df = pd.concat([all_results_df, temp_df], ignore_index=True)

print("Processamento de arquivos concluído.")

# --- CONSTRUÇÃO E CORREÇÃO DA TABELA FINAL ---
final_table = all_results_df[[
    'Algorithm', 'Set', group_column, 
    'Count_0', 'Perc_0', 'Count_1', 'Perc_1', 
    'Total_Count'
]].copy()

final_table.columns = [
    'Algoritmo', 'Conjunto', 'Cluster', 
    'Contagem (0)', '% (0)', 'Contagem (1)', '% (1)', 
    'Total'
]

# 1. Ordena a tabela
final_table = final_table.sort_values(by=['Algoritmo', 'Conjunto', 'Cluster'])

# 2. **CORREÇÃO NA AMBIGUIDADE DOS DADOS (O MAIS PROVÁVEL) E DO ÍNDICE**
# Preenche NaNs com 0 para evitar que o .to_latex() falhe ao formatar
colunas_numericas = ['Contagem (0)', '% (0)', 'Contagem (1)', '% (1)', 'Total']
final_table[colunas_numericas] = final_table[colunas_numericas].fillna(0)

# Força um índice simples
final_table = final_table.reset_index(drop=True)


# --- VISUALIZAÇÃO (Markdown) ---
final_table_markdown = final_table.copy()
final_table_markdown['% (0)'] = final_table_markdown['% (0)'].map('{:.2f}%'.format)
final_table_markdown['% (1)'] = final_table_markdown['% (1)'].map('{:.2f}%'.format)

print("\n--- Visualização da Tabela (Markdown para conferência) ---")
print(final_table_markdown.to_markdown(index=False, floatfmt=(".0f")))
print("----------------------------------------------------------")


# --- CÓDIGO DA CÉLULA 2: GERAÇÃO DO LATEX ---

print("\n--- Geração do CÓDIGO LATEX FORMAL ---")

# 1. O dicionário de formatadores
formatadores_latex = {
    'Contagem (0)': '{:.0f}'.format,
    '% (0)': '{:.2f}'.format,
    'Contagem (1)': '{:.0f}'.format,
    '% (1)': '{:.2f}'.format,
    'Total': '{:.0f}'.format
}

# 2. Gera o CORPO da tabela em LaTeX
latex_corpo_tabela = final_table.to_latex(
    buf=None, 
    columns=final_table.columns, 
    header=False,
    index=False,
    column_format='llc | rr | rr | r', 
    formatters=formatadores_latex,    
    escape=False 
).strip() 

# 3. Template LaTeX
latex_template_completo = r"""
% --- Início do Código LaTeX Gerado ---
% Pacotes necessários no seu preâmbulo .tex: \usepackage{booktabs}, \usepackage{multirow}, \usepackage{adjustbox}
\begin{table}[!htbp]
    \centering
    \caption{Distribuição dos Rótulos (Eye State) em Cada Cluster por Algoritmo}
    \label{tab:distribuicao_clusters}
    \adjustbox{maxwidth=\textwidth}{
    \begin{tabular}{llc | rr | rr | r}
        \toprule
        % Cabeçalho multilinha
        \multicolumn{3}{c|}{\bfseries Dados do Cluster} & \multicolumn{2}{c|}{\bfseries Rótulo 0 (Olhos Fechados)} & \multicolumn{2}{c|}{\bfseries Rótulo 1 (Olhos Abertos)} & \multicolumn{1}{c}{\multirow{2}{*}{\bfseries Total}} \\
        \cmidrule(lr){1-3} \cmidrule(lr){4-5} \cmidrule(lr){6-7}
        \bfseries Algoritmo & \bfseries Conjunto & \bfseries Cluster & \bfseries Contagem & \bfseries (\%) & \bfseries Contagem & \bfseries (\%) & \\
        \midrule
{corpo_da_tabela}
        \bottomrule
    \end{tabular}
    } % Fim do adjustbox
    \footnotesize Fonte: Processamento dos resultados da clusterização.
\end{table}
% --- Fim do Código LaTeX Gerado ---
"""

# 4. Combina o corpo com o template e imprime o resultado final
codigo_latex_final = latex_template_completo.format(
    corpo_da_tabela=latex_corpo_tabela
)

print(codigo_latex_final)
print("------------------------------------\n")

In [6]:
import pandas as pd
import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mlex import DataReader, PreProcessingTransformer, PastFutureSplit

# --- CONFIGURAÇÃO ---
base_path = r'/data/eeg_eyestate'
cluster_names = ['kmeans', 'gmm', 'agglomerative']
sets = ['train', 'test']
target_column = 'eyeDetection'
# Define o diretório onde os PDFs serão salvos
OUTPUT_DIR = 'correlation_heatmaps_pearson' 
os.makedirs(OUTPUT_DIR, exist_ok=True) # Cria a pasta se ela não existir
GROUP_CATEGORIES = [np.array([0, 1, 2], dtype=int)]
correlation_matrices = {}

print("--- 1. Iniciando o processamento: Cálculo das Matrizes de Correlação ---")

for cluster_name in cluster_names:
    for data_set in sets:
        file_name = f'EEG_Eye_State_{cluster_name}_{data_set}.csv'
        path = os.path.join(base_path, file_name)
        
        print(f"  > Processando {file_name}...")
        
        try:
            reader = DataReader(path, target_columns=[target_column] )
            X,y = reader.get_X_y()

            if 'GROUP' in X.columns:
                X['GROUP'] = X['GROUP'].astype('category')
                
            numeric_cols_for_preprocessing = [col for col in X.columns.to_list() if col not in ['Timestamp','GROUP']]
            preprocessor = PreProcessingTransformer(target_column=[target_column],numeric_features=numeric_cols_for_preprocessing,categorical_features=['GROUP'],categories=GROUP_CATEGORIES,handle_unknown='ignore')

            preprocessor.fit(X, y)
            X_array, y_array = preprocessor.transform(X, y)

            df_processed = pd.DataFrame(
                X_array,
                columns=preprocessor.get_feature_names_out() # Nomes corretos das colunas transformadas
            )
            
            df_processed[target_column] = y_array.flatten()

            correlation_matrix = df_processed.corr(method='pearson')
            
            key = f'{cluster_name}_{data_set}'
            correlation_matrices[key] = correlation_matrix
        except FileNotFoundError:
            print(f"  ⚠️ Aviso: Arquivo não encontrado em {path}. Pulando.")
            continue
        except Exception as e:
            print(f"  ❌ Erro ao processar {file_name}: {e}")
            continue


print(f"\n--- 2. Geração dos Mapas de Calor (Heatmaps) e salvamento em PDF na pasta '{OUTPUT_DIR}' ---")

if correlation_matrices:
    for key, matrix in correlation_matrices.items():
        
        # Cria a máscara para o triângulo inferior
        mask = np.triu(np.ones_like(matrix, dtype=bool), k=0) 

        plt.figure(figsize=(10, 8))
        
        title = f'Pearson Correlation Heatmap for {key.upper()}'
        plt.title(title, fontsize=16)
        
        sns.heatmap(
            matrix,
            mask=mask,
            annot=True,
            fmt=".2f",
            cmap='coolwarm',
            linewidths=.5,
            linecolor='black',
            cbar_kws={'label': r'Pearson Correlation Coefficient ($\rho$)'}
        )
        
        plt.xticks(rotation=90)
        plt.yticks(rotation=0) 

        plt.tight_layout()
        
        # --- MUDANÇA PRINCIPAL: SALVAR COMO PDF EM VEZ DE MOSTRAR ---
        output_filename = os.path.join(OUTPUT_DIR, f'correlation_{key}.pdf')
        plt.savefig(output_filename, bbox_inches='tight')
        plt.close() # Fecha a figura para liberar memória
        
        print(f"  Arquivo salvo: {output_filename}")
        
else:
    print("Não foi possível gerar mapas de calor: Nenhuma matriz de correlação foi calculada com sucesso.")

print(f"\n--- Processamento concluído. Todos os PDFs estão na pasta '{OUTPUT_DIR}'. ---")

--- 1. Iniciando o processamento: Cálculo das Matrizes de Correlação ---
  > Processando EEG_Eye_State_kmeans_train.csv...
  > Processando EEG_Eye_State_kmeans_test.csv...
  > Processando EEG_Eye_State_gmm_train.csv...
  > Processando EEG_Eye_State_gmm_test.csv...
  > Processando EEG_Eye_State_agglomerative_train.csv...
  > Processando EEG_Eye_State_agglomerative_test.csv...

--- 2. Geração dos Mapas de Calor (Heatmaps) e salvamento em PDF na pasta 'correlation_heatmaps_pearson' ---
  Arquivo salvo: correlation_heatmaps_pearson/correlation_kmeans_train.pdf
  Arquivo salvo: correlation_heatmaps_pearson/correlation_kmeans_test.pdf
  Arquivo salvo: correlation_heatmaps_pearson/correlation_gmm_train.pdf
  Arquivo salvo: correlation_heatmaps_pearson/correlation_gmm_test.pdf
  Arquivo salvo: correlation_heatmaps_pearson/correlation_agglomerative_train.pdf
  Arquivo salvo: correlation_heatmaps_pearson/correlation_agglomerative_test.pdf

--- Processamento concluído. Todos os PDFs estão na past